# Building an MLP in PyTorch: Stock Return Prediction

**Chapter 6 — Deep Neural Networks** | Python Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/Bishop_Ch_06/NB_bishop_ch06_mlp_demo.ipynb)

This notebook builds a two-hidden-layer MLP in PyTorch to predict daily S&P 500 returns
from lagged returns and rolling volatility, then benchmarks it against a linear (OLS) baseline.

## 1. Install Dependencies

In [ ]:
!pip install -q torch yfinance

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import yfinance as yf
from pathlib import Path

# ── reproducibility ──────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── course colours ───────────────────────────────────────────────
BLUE    = "#1f4e79"
CRIMSON = "#c00000"
GREEN   = "#2e7d32"

# ── matplotlib defaults ──────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "none",
    "axes.facecolor":   "none",
    "savefig.facecolor": "none",
    "font.size": 12,
})

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

def save_fig(fig, name, dpi=150):
    """Save a figure with transparent background."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight",
                transparent=True)
    print(f"Saved {name}.png")

## 3. Load S&P 500 Data

In [ ]:
# Download daily S&P 500 data (2015-2024)
sp500 = yf.download("^GSPC", start="2015-01-01", end="2024-12-31",
                     auto_adjust=True, progress=False)

# Compute simple daily returns
sp500["Return"] = sp500["Close"].pct_change()

# Create lagged features r_{t-1} ... r_{t-5}
for lag in range(1, 6):
    sp500[f"Return_lag{lag}"] = sp500["Return"].shift(lag)

# 20-day rolling volatility (annualised)
sp500["RollingVol_20"] = sp500["Return"].rolling(20).std() * np.sqrt(252)

# Drop rows with NaN and keep only features + target
feature_cols = [f"Return_lag{i}" for i in range(1, 6)] + ["RollingVol_20"]
target_col   = "Return"

data = sp500[feature_cols + [target_col]].dropna().copy()
print(f"Dataset shape: {data.shape}")
print(f"Date range: {data.index[0].date()} to {data.index[-1].date()}")
data.head()

## 4. Train / Test Split

In [ ]:
from sklearn.preprocessing import StandardScaler

# Chronological 80/20 split
split_idx = int(len(data) * 0.8)

train_df = data.iloc[:split_idx]
test_df  = data.iloc[split_idx:]

print(f"Train: {len(train_df)} rows  ({train_df.index[0].date()} to {train_df.index[-1].date()})")
print(f"Test:  {len(test_df)} rows  ({test_df.index[0].date()} to {test_df.index[-1].date()})")

# Standardise features (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values)
X_test  = scaler.transform(test_df[feature_cols].values)

y_train = train_df[target_col].values.reshape(-1, 1)
y_test  = test_df[target_col].values.reshape(-1, 1)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

print(f"\nFeature tensor shape: {X_train_t.shape}")
print(f"Target  tensor shape: {y_train_t.shape}")

## 5. Build the MLP

In [ ]:
n_features = X_train_t.shape[1]

mlp = nn.Sequential(
    nn.Linear(n_features, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

print(mlp)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

## 6. Train the MLP

In [ ]:
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
criterion = nn.MSELoss()

n_epochs = 200
train_losses = []
test_losses  = []

for epoch in range(1, n_epochs + 1):
    # ── train step ───────────────────────────────────────────────
    mlp.train()
    optimizer.zero_grad()
    pred_train = mlp(X_train_t)
    loss_train = criterion(pred_train, y_train_t)
    loss_train.backward()
    optimizer.step()

    # ── evaluate ─────────────────────────────────────────────────
    mlp.eval()
    with torch.no_grad():
        pred_test = mlp(X_test_t)
        loss_test = criterion(pred_test, y_test_t)

    train_losses.append(loss_train.item())
    test_losses.append(loss_test.item())

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:>4d}/{n_epochs}  "
              f"Train MSE: {loss_train.item():.6f}  "
              f"Test MSE: {loss_test.item():.6f}")

## 7. Evaluate: RMSE, Directional Accuracy & OLS Baseline

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ── MLP predictions ──────────────────────────────────────────────
mlp.eval()
with torch.no_grad():
    y_pred_mlp = mlp(X_test_t).numpy().flatten()

y_true = y_test.flatten()

rmse_mlp = np.sqrt(mean_squared_error(y_true, y_pred_mlp))
dir_acc_mlp = np.mean(np.sign(y_pred_mlp) == np.sign(y_true))

# ── OLS baseline ─────────────────────────────────────────────────
ols = LinearRegression().fit(X_train, y_train.ravel())
y_pred_ols = ols.predict(X_test).flatten()

rmse_ols = np.sqrt(mean_squared_error(y_true, y_pred_ols))
dir_acc_ols = np.mean(np.sign(y_pred_ols) == np.sign(y_true))

# ── summary table ────────────────────────────────────────────────
results = pd.DataFrame({
    "Model":               ["OLS (Linear)", "MLP (2-layer)"],
    "Test RMSE":           [rmse_ols, rmse_mlp],
    "Directional Accuracy": [f"{dir_acc_ols:.1%}", f"{dir_acc_mlp:.1%}"],
})
print(results.to_string(index=False))

## 8. Visualisations

In [ ]:
# ── 8a. Training loss curve ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, color=BLUE,    linewidth=1.5, label="Train MSE")
ax.plot(test_losses,  color=CRIMSON, linewidth=1.5, label="Test MSE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("MLP Training & Test Loss", fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2,
          frameon=False)
fig.tight_layout()
save_fig(fig, "mlp_loss_curve")
plt.show()

In [ ]:
# ── 8b. Actual vs Predicted returns (last 120 trading days) ──────
n_show = 120
dates = test_df.index[-n_show:]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates, y_true[-n_show:],    color=BLUE,    linewidth=1.0,
        alpha=0.7, label="Actual")
ax.plot(dates, y_pred_mlp[-n_show:], color=CRIMSON, linewidth=1.0,
        alpha=0.7, label="MLP Predicted")
ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Return")
ax.set_title("Actual vs MLP-Predicted Returns (last 120 days)",
             fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2,
          frameon=False)
fig.tight_layout()
save_fig(fig, "mlp_actual_vs_predicted")
plt.show()

In [ ]:
# ── 8c. Feature importance via input gradients ───────────────────
mlp.eval()
X_test_grad = X_test_t.clone().requires_grad_(True)
out = mlp(X_test_grad)
out.sum().backward()

importance = X_test_grad.grad.abs().mean(dim=0).numpy()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(feature_cols, importance, color=BLUE, edgecolor="white")
ax.set_xlabel("Mean |Gradient|")
ax.set_title("Feature Importance (Input Gradients)", fontweight="bold")
ax.invert_yaxis()
fig.tight_layout()
save_fig(fig, "mlp_feature_importance")
plt.show()

## 9. Business Interpretation: Sharpe Ratio & Transaction Costs

In [ ]:
# ── Strategy: go long when MLP predicts positive return, else cash ─
signal_mlp = np.where(y_pred_mlp > 0, 1.0, 0.0)
signal_ols = np.where(y_pred_ols > 0, 1.0, 0.0)

# Gross strategy returns
strat_mlp_gross = signal_mlp * y_true
strat_ols_gross = signal_ols * y_true

# Transaction costs: 5 bps per trade (when signal changes)
tc_bps = 5e-4
trades_mlp = np.abs(np.diff(signal_mlp, prepend=0))
trades_ols = np.abs(np.diff(signal_ols, prepend=0))

strat_mlp_net = strat_mlp_gross - trades_mlp * tc_bps
strat_ols_net = strat_ols_gross - trades_ols * tc_bps

def annualised_sharpe(returns, risk_free_daily=0.0):
    excess = returns - risk_free_daily
    if excess.std() == 0:
        return 0.0
    return (excess.mean() / excess.std()) * np.sqrt(252)

sharpe_bh  = annualised_sharpe(y_true)
sharpe_mlp = annualised_sharpe(strat_mlp_net)
sharpe_ols = annualised_sharpe(strat_ols_net)

cum_bh  = (1 + y_true).cumprod()
cum_mlp = (1 + strat_mlp_net).cumprod()
cum_ols = (1 + strat_ols_net).cumprod()

print(f"{'Strategy':<20} {'Sharpe':>8}  {'Cumulative Return':>18}")
print("-" * 50)
print(f"{'Buy & Hold':<20} {sharpe_bh:>8.2f}  {cum_bh[-1] - 1:>17.2%}")
print(f"{'OLS Long/Cash':<20} {sharpe_ols:>8.2f}  {cum_ols[-1] - 1:>17.2%}")
print(f"{'MLP Long/Cash':<20} {sharpe_mlp:>8.2f}  {cum_mlp[-1] - 1:>17.2%}")
print(f"\nTransaction cost assumption: {tc_bps*1e4:.0f} bps per trade")

In [ ]:
# ── Cumulative returns plot ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
dates_test = test_df.index

ax.plot(dates_test, cum_bh,  color="grey",  linewidth=1.2, label="Buy & Hold")
ax.plot(dates_test, cum_ols, color=GREEN,   linewidth=1.2, label="OLS Long/Cash")
ax.plot(dates_test, cum_mlp, color=CRIMSON, linewidth=1.5, label="MLP Long/Cash")
ax.set_ylabel("Cumulative Return (\$1 invested)")
ax.set_title("Strategy Comparison (net of 5 bps transaction costs)",
             fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3,
          frameon=False)
fig.tight_layout()
save_fig(fig, "mlp_strategy_comparison")
plt.show()

### Interpretation

The table above shows whether the MLP's statistical improvement over OLS translates into
economically meaningful alpha after realistic transaction costs.  Key observations:

- **Directional accuracy** above 50 % is necessary but not sufficient for profitability.
- **Transaction costs erode** returns quickly when the model trades frequently (the MLP
  signal can flip daily).
- The **Sharpe ratio** gives a risk-adjusted view; a strategy Sharpe below ~0.5 is hard
  to justify in practice.
- Non-linear models shine more when features include richer cross-sectional or alternative
  data, not just five lagged returns.

## 10. Key Takeaways

1. **MLPs with ReLU activations** can capture non-linear relationships between lagged
   returns and future returns, but the improvement over a linear model is typically small
   for univariate price data.

2. **Chronological train/test splits** are essential in finance to avoid look-ahead bias;
   random splits would grossly overstate performance.

3. **Feature engineering matters more than architecture**: rolling volatility and lagged
   returns are a minimal feature set. Richer inputs (sentiment, macro indicators,
   cross-sectional features) typically yield larger gains.

4. **Statistical vs economic significance**: a lower RMSE does not guarantee trading
   profits once transaction costs, slippage, and capacity constraints are considered.

5. **Gradient-based feature importance** provides a model-agnostic way to understand
   which inputs the network relies on, bridging the gap between black-box predictions
   and business insight.